## Concept 6 — RAG Agent

### What is it?
A `create_agent`-based agent where one of the tools is a FAISS vectorstore retriever.
The agent decides *when* to search the vectorstore — only when the query actually needs it.

### Fixed RAG Pipeline vs RAG Agent
```
Fixed RAG Pipeline (always retrieves):
  Query → Retrieve → LLM → Answer
  Even 'What is 2+2?' retrieves documents — wasteful.

RAG Agent (decides when to retrieve):
  'What is 2+2?'   → Agent calls calculate, skips vectorstore
  'What is RAG?'   → Agent calls search_docs_rag (vectorstore)
  'Docs + math'    → Agent calls both
  Smarter and more efficient.
```

### Pipeline
```
Query → create_agent (tools: search_docs_rag, calculate, get_weather)
  → agent decides: retrieve? calculate? both?
  → executes chosen tools
  → Final Answer grounded in real retrieved docs
```

### Limitation (what Concept 7 fixes)
One agent handles all tasks — no specialization.
→ Concept 7 routes to specialized sub-agents, each tuned for one task type.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, calculate, get_weather, TEST_QUERIES, run_queries
from langchain.agents import create_agent
from langchain_core.tools import tool

llm = get_llm()
print('LLM ready — building RAG vectorstore...')

### Step 1 — Build FAISS Vectorstore
Load real LangSmith docs, chunk and embed into FAISS.

In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

loader    = WebBaseLoader('https://docs.langchain.com/langsmith/agent-server')
docs      = loader.load()
chunks    = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
retriever = FAISS.from_documents(chunks, OpenAIEmbeddings(model='text-embedding-3-small')).as_retriever(search_kwargs={'k': 3})

print(f'Vectorstore ready: {len(docs)} page → {len(chunks)} chunks')

### Step 2 — Wrap Retriever as a Tool
The @tool decorator makes the retriever available to the agent like any other tool.

In [ ]:
@tool
def search_docs_rag(query: str) -> str:
    """Search the LangSmith documentation for information about agents, RAG, deployment, and LangChain concepts."""
    results = retriever.invoke(query)
    if not results:
        return 'No relevant documentation found.'
    return '\n\n'.join(doc.page_content[:400] for doc in results)

rag_tools = [search_docs_rag, calculate, get_weather]
print(f'RAG agent tools: {[t.name for t in rag_tools]}')

### Step 3 — Create the RAG Agent
`create_agent` with the RAG tool included. The agent decides autonomously when to use it.

In [ ]:
rag_agent = create_agent(
    model=llm,
    tools=rag_tools,
    system_prompt=(
        'You are a helpful assistant. '
        'For LangChain/AI topics use search_docs_rag. '
        'For math use calculate. For weather use get_weather. '
        'Do not guess — always use the appropriate tool.'
    ),
)
print('RAG agent created')

### Step 4 — See When the Agent Searches vs Calculates
Watch how the agent picks the right tool for each query type.

In [ ]:
test_cases = [
    ('Math only',    'What is 144 divided by 12?'),
    ('Docs only',    'What is LangSmith used for?'),
    ('Weather only', 'What is the weather in Bangalore?'),
    ('Mixed',        'Explain LangSmith and also calculate 8 times 7'),
]

for label, q in test_cases:
    print(f'\n[{label}]  Q: {q}')
    result = rag_agent.invoke({'messages': [{'role': 'user', 'content': q}]})

    tools_called = []
    for msg in result['messages']:
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for call in msg.tool_calls:
                tools_called.append(call['name'])

    answer = result['messages'][-1].content
    print(f'  Tools: {tools_called}')
    print(f'  A: {answer[:180]}...' if len(answer) > 180 else f'  A: {answer}')
    print('-' * 50)

### Full Function

In [ ]:
def run_rag_agent(query: str) -> str:
    result = rag_agent.invoke({'messages': [{'role': 'user', 'content': query}]})
    return result['messages'][-1].content

### Test — Standard Queries

In [ ]:
run_queries(run_rag_agent, label='Concept 6 — RAG Agent')